# Определение частей речи: обучение на SynTagRus

Ноутбук развивает `pos_tagging.ipynb`. Цели работы:

1. Перевести обучение на ручную разметку **SynTagRus** (формат Universal Dependencies, UPOS).
2. Кэшировать данные и обученные модели, чтобы не пересчитывать всё при каждом запуске.
3. Оценить модели на корпусе **nerus_lenta** (новости), который не участвовал в обучении.
4. Оставить **10 морфологических признаков** и сделать многозадачную модель для сравнения.
5. Проверить модели на омонимичных словах.
6. Дать блок, в котором можно ввести слово или предложение и получить разбор.

Каждой цели соответствует отдельный раздел ниже.

## Кэширование и настройки

Что сохраняется на диск, чтобы не повторять долгие шаги:

- `cache/` — разобранные корпуса (CoNLL-U → списки токенов). Разбор SynTagRus занимает время, поэтому результат записывается в pickle.
- `models_syntagrus/` — обученные модели `*.keras`, история обучения `*_history.json` и словарь `vocab.json` (символы, значения признаков, список тегов).

При повторном запуске ноутбук загружает модели из кэша. Чтобы переобучить всё, поставьте `USE_CACHE = False`.
Переменная `SMOKE = True` (или переменная окружения `SMOKE=1`) запускает быструю проверку кода на малой выборке: она нужна только для отладки.

In [1]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import re, csv, json, gzip, pickle, random, collections
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

SMOKE = os.environ.get('SMOKE') == '1'
USE_CACHE = True

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(os.path.dirname(BASE_DIR), 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'cache_smoke' if SMOKE else 'cache')
MODEL_DIR = os.path.join(BASE_DIR, 'models_smoke' if SMOKE else 'models_syntagrus')
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

SYNTAGRUS_PATH = os.path.join(DATA_DIR, 'ru_syntagrus-ud-2.0-morpho-train.txt')
NERUS_PATH = os.path.join(DATA_DIR, 'nerus_lenta.conllu.gz')
CSV_PATH = os.path.join(DATA_DIR, 'ruscorpora_content.csv')

MAX_WORDS = 40          # слов в предложении (95-й перцентиль длины в SynTagRus — 37)
MAX_CHARS = 20          # символов в слове
CHAR_EMB_DIM = 32
N_FILTERS = 64
KERNEL_SIZES = (3, 4, 5)
N_MORPH = 10            # число морфологических признаков в многозадачной модели
EPOCHS = 2 if SMOKE else 15
BATCH_SIZE = 64
SYNTAGRUS_SENTS = 3000 if SMOKE else None
NERUS_SENTS = 500 if SMOKE else 10000

UPOS = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON',
        'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
UPOS2ID = {t: i for i, t in enumerate(UPOS)}
PAD_LABEL = -1          # метка для дополненных позиций; в loss и accuracy не учитывается
PAD_ID, UNK_ID = 0, 1
TOKEN_RE = re.compile(r'\w+(?:-\w+)*|[^\w\s]')

print('SMOKE:', SMOKE, '| кэш:', CACHE_DIR, '| модели:', MODEL_DIR)

SMOKE: True | кэш: C:\Users\reino\Desktop\Учеба\Homework\NLP\pos_tagging\cache_smoke | модели: C:\Users\reino\Desktop\Учеба\Homework\NLP\pos_tagging\models_smoke


## Цель 1. Обучение на ручной разметке SynTagRus

**Что сделано**

- Загружен корпус `ru_syntagrus-ud-2.0-morpho-train.txt` (CoNLL-U): 54 529 предложений, около 962 тыс. токенов.
  Части речи размечены вручную, поэтому эталон не зависит от анализатора.
- Набор тегов — 17 UPOS (вместо 12 тегов pymorphy). Появились `PROPN`, `AUX`, `SCONJ`, `DET`, `SYM`.
- Разбиение: 95% предложений — обучение, 5% — валидация. Официальных dev/test в папке нет, поэтому отдельный тест — `nerus_lenta` (цель 2).
- Длинные предложения режутся на куски по `MAX_WORDS = 40` слов, короткие дополняются `PAD`.

**Выгоды.** Обучающих данных примерно в 300 раз больше, чем в прошлой версии (3 200 токенов). Модель учится на
человеческой разметке, а не на ответах анализатора, поэтому её точность — честная оценка.

In [2]:
def read_conllu(path, max_sents=None):
    opener = gzip.open if path.endswith('.gz') else open
    sents, cur = [], []
    with opener(path, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                if cur:
                    sents.append(cur)
                    cur = []
                    if max_sents and len(sents) >= max_sents:
                        break
                continue
            if line.startswith('#'):
                continue
            cols = line.split('\t')
            if '-' in cols[0] or '.' in cols[0]:
                continue
            feats = {} if cols[5] == '_' else dict(kv.split('=', 1) for kv in cols[5].split('|'))
            cur.append((cols[1], cols[3], feats))
    if cur and (not max_sents or len(sents) < max_sents):
        sents.append(cur)
    return sents


def cached(name, builder):
    path = os.path.join(CACHE_DIR, name + '.pkl')
    if USE_CACHE and os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    obj = builder()
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    return obj


syntagrus = cached('syntagrus', lambda: read_conllu(SYNTAGRUS_PATH, SYNTAGRUS_SENTS))
nerus = cached('nerus_lenta', lambda: read_conllu(NERUS_PATH, NERUS_SENTS))

random.Random(SEED).shuffle(syntagrus)
n_val = int(0.05 * len(syntagrus))
val_sents, train_sents = syntagrus[:n_val], syntagrus[n_val:]
print(f'SynTagRus: предложений {len(syntagrus)}, обучение {len(train_sents)}, валидация {len(val_sents)}')
print(f'nerus_lenta: предложений {len(nerus)}, токенов {sum(len(s) for s in nerus)}')

SynTagRus: предложений 3000, обучение 2850, валидация 150
nerus_lenta: предложений 500, токенов 8920


In [3]:
upos_cnt = collections.Counter(u for s in train_sents for _, u, _ in s)
print('Токенов в обучении:', sum(upos_cnt.values()))
print('Части речи:', dict(upos_cnt.most_common()))

lens = sorted(len(s) for s in train_sents)
print('Длина предложения: медиана', lens[len(lens) // 2], '| 95-й перцентиль', lens[int(0.95 * len(lens))])
word_lens = sorted(len(f) for s in train_sents for f, _, _ in s)
print('Длина слова: 99-й перцентиль', word_lens[int(0.99 * len(word_lens))], '| максимум', word_lens[-1])

Токенов в обучении: 41938
Части речи: {'NOUN': 9417, 'PUNCT': 8429, 'VERB': 5633, 'ADP': 3711, 'ADJ': 3424, 'ADV': 2350, 'PRON': 2125, 'PART': 1513, 'CCONJ': 1445, 'PROPN': 1437, 'DET': 1043, 'SCONJ': 705, 'NUM': 423, 'AUX': 249, 'SYM': 23, 'INTJ': 11}
Длина предложения: медиана 13 | 95-й перцентиль 33
Длина слова: 99-й перцентиль 14 | максимум 30


In [4]:
def to_chunks(sents):
    return [s[i:i + MAX_WORDS] for s in sents for i in range(0, len(s), MAX_WORDS)]


def forms_of(chunks):
    return [[f for f, _, _ in ch] for ch in chunks]


train_chunks = to_chunks(train_sents)
val_chunks = to_chunks(val_sents)
test_chunks = to_chunks(nerus)

chars = sorted({c for ch in train_chunks for f, _, _ in ch for c in f})
char2id = {c: i + 2 for i, c in enumerate(chars)}
N_CHARS = len(char2id) + 2


def encode_forms(chunks):
    X = np.zeros((len(chunks), MAX_WORDS, MAX_CHARS), dtype='int32')
    for i, ch in enumerate(chunks):
        for j, form in enumerate(ch):
            for k, c in enumerate(form[:MAX_CHARS]):
                X[i, j, k] = char2id.get(c, UNK_ID)
    return X


def encode_targets(chunks, morph_keys=()):
    y = {'upos': np.full((len(chunks), MAX_WORDS), PAD_LABEL, dtype='int32')}
    for k in morph_keys:
        y['morph_' + k] = np.full((len(chunks), MAX_WORDS), PAD_LABEL, dtype='int32')
    for i, ch in enumerate(chunks):
        for j, (_, upos, feats) in enumerate(ch):
            y['upos'][i, j] = UPOS2ID[upos]
            for k in morph_keys:
                if k in feats:
                    y['morph_' + k][i, j] = MORPH_VAL2ID[k].get(feats[k], PAD_LABEL)
    return y


X_train, X_val, X_test = encode_forms(forms_of(train_chunks)), encode_forms(forms_of(val_chunks)), encode_forms(forms_of(test_chunks))
y_train, y_val, y_test = encode_targets(train_chunks), encode_targets(val_chunks), encode_targets(test_chunks)
print('X_train:', X_train.shape, '| символьный словарь:', N_CHARS)

X_train: (2898, 40, 20) | символьный словарь: 112


### Модели одной задачи (только UPOS)

- **CNN:** Embedding символов (32) → свёртки по символам с ядрами 3, 4, 5 (по 64 фильтра) и max-pooling → вектор слова (192) → свёртка по словам (ядро 3, 128 фильтров) → Dense(64) → softmax по 17 тегам.
- **RNN:** Embedding символов → BiLSTM(64) по символам слова → BiLSTM(128) по словам → softmax. Рекуррентная сеть нужна по условию задания, поэтому оставлена для сравнения.

Обучение: Adam (lr 1e-3), batch 64, до 15 эпох, EarlyStopping по `val_loss` с patience 3. Потери и accuracy считаются только по реальным словам.

In [5]:
def masked_loss(y_true, y_pred):
    mask = tf.cast(y_true != PAD_LABEL, tf.float32)
    ce = keras.losses.sparse_categorical_crossentropy(tf.maximum(y_true, 0), y_pred)
    return tf.reduce_sum(ce * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)


def masked_accuracy(y_true, y_pred):
    mask = tf.cast(y_true != PAD_LABEL, tf.float32)
    hit = tf.cast(tf.equal(tf.maximum(y_true, 0), tf.argmax(y_pred, axis=-1, output_type=tf.int32)), tf.float32)
    return tf.reduce_sum(hit * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)


def build_model(kind, morph=False):
    char_in = keras.Input(shape=(MAX_CHARS,), dtype='int32')
    if kind == 'cnn':
        x = layers.Embedding(N_CHARS, CHAR_EMB_DIM)(char_in)
        pooled = [layers.GlobalMaxPooling1D()(layers.Conv1D(N_FILTERS, k, activation='relu', padding='same')(x))
                  for k in KERNEL_SIZES]
        word_vec = layers.Dropout(0.3)(layers.Concatenate()(pooled))
    else:
        x = layers.Embedding(N_CHARS, CHAR_EMB_DIM, mask_zero=True)(char_in)
        word_vec = layers.Bidirectional(layers.LSTM(64))(x)
    word_encoder = keras.Model(char_in, word_vec)

    sent_in = keras.Input(shape=(MAX_WORDS, MAX_CHARS), dtype='int32')
    h = layers.TimeDistributed(word_encoder)(sent_in)
    if kind == 'cnn':
        h = layers.Conv1D(128, 3, activation='relu', padding='same')(h)
        h = layers.TimeDistributed(layers.Dense(64, activation='relu'))(h)
    else:
        h = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(h)
    h = layers.Dropout(0.3)(h)

    outputs = {'upos': layers.TimeDistributed(layers.Dense(len(UPOS), activation='softmax'), name='upos')(h)}
    if morph:
        for k in MORPH_KEYS:
            name = 'morph_' + k
            outputs[name] = layers.TimeDistributed(
                layers.Dense(len(MORPH_VAL2ID[k]), activation='softmax'), name=name)(h)
    return keras.Model(sent_in, outputs)


def compile_and_train(name, model, X_tr, y_tr, X_va, y_va):
    path = os.path.join(MODEL_DIR, name + '.keras')
    hist_path = os.path.join(MODEL_DIR, name + '_history.json')
    if USE_CACHE and os.path.exists(path):
        print(f'{name}: загружена из кэша')
        model = keras.models.load_model(path, compile=False)
        with open(hist_path, encoding='utf-8') as f:
            return model, json.load(f)

    weights = {n: (1.0 if n == 'upos' else 0.2) for n in model.output_names}
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss={n: masked_loss for n in model.output_names},
        loss_weights=weights,
        metrics={'upos': [masked_accuracy]},
    )
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
    h = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=EPOCHS,
                  batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=2)
    model.save(path)
    history = {k: [float(x) for x in v] for k, v in h.history.items()}
    with open(hist_path, 'w', encoding='utf-8') as f:
        json.dump(history, f)
    return model, history


def make_and_train(name, kind, morph=False):
    model = build_model(kind, morph)
    ytr, yva = (y_train_m, y_val_m) if morph else (y_train, y_val)
    return compile_and_train(name, model,
                             X_train, {n: ytr[n] for n in model.output_names},
                             X_val, {n: yva[n] for n in model.output_names})


def predict_dict(model, X):
    out = model.predict(X, batch_size=256, verbose=0)
    if not isinstance(out, dict):
        out = dict(zip(model.output_names, out if isinstance(out, list) else [out]))
    return out


cnn_single, hist_cnn = make_and_train('cnn_single', 'cnn')
rnn_single, hist_rnn = make_and_train('rnn_single', 'rnn')

Epoch 1/2


46/46 - 13s - 293ms/step - loss: 2.1889 - masked_accuracy: 0.3212 - val_loss: 1.5724 - val_masked_accuracy: 0.4387


Epoch 2/2


46/46 - 1s - 32ms/step - loss: 1.4548 - masked_accuracy: 0.5070 - val_loss: 1.1117 - val_masked_accuracy: 0.6284


Restoring model weights from the end of the best epoch: 2.


Epoch 1/2


46/46 - 144s - 3s/step - loss: 2.1596 - masked_accuracy: 0.3262 - val_loss: 1.5487 - val_masked_accuracy: 0.4497


Epoch 2/2


46/46 - 74s - 2s/step - loss: 1.4445 - masked_accuracy: 0.4926 - val_loss: 1.2897 - val_masked_accuracy: 0.5301


Restoring model weights from the end of the best epoch: 2.


## Цель 2. Оценка на nerus_lenta

**Что такое nerus_lenta.** Это корпус новостей Lenta.ru из набора NeRus. Файл — CoNLL-U с UPOS, морфологией и NER-метками,
весит около 2 ГБ в сжатом виде. Для теста берутся первые 10 000 предложений (около 180 тыс. токенов).

**Почему выбран.**
- Его нет в обучении, поэтому оценка показывает переносимость модели на новые тексты.
- Разметка тоже в стиле UD (UPOS), значит, метки совместимы с обучением.
- Он другой по источнику и стилю (новостной текст), что честнее проверяет модель, чем разбиение внутри SynTagRus.
- Объём большой, поэтому метрики устойчивы. Файл читается потоково и обрезается по числу предложений, полностью в память не загружается.
- NER-метки в этой работе не используются. Их можно добавить как вторую задачу в будущем.

In [6]:
results, reports = {}, {}


def evaluate_upos(name, model):
    pred = predict_dict(model, X_test)['upos'].argmax(-1)
    mask = y_test['upos'] != PAD_LABEL
    results[name] = pred
    rep = classification_report(y_test['upos'][mask], pred[mask], labels=range(len(UPOS)),
                                target_names=UPOS, zero_division=0, output_dict=True)
    reports[name] = rep
    return {'точность': (pred[mask] == y_test['upos'][mask]).mean(), 'macro F1': rep['macro avg']['f1-score']}


scores = {}
scores['CNN (одна задача)'] = evaluate_upos('CNN (одна задача)', cnn_single)
scores['RNN (одна задача)'] = evaluate_upos('RNN (одна задача)', rnn_single)
pd.DataFrame(scores).T.round(4)

,точность,macro F1
CNN (одна задача),0.6159,0.2653
RNN (одна задача),0.5419,0.1856


In [7]:
cnn_rep = pd.DataFrame(reports['CNN (одна задача)']).T.loc[UPOS].round(2)
cnn_rep

,precision,recall,f1-score,support
ADJ,0.54,0.22,0.31,884.0
ADP,0.56,0.94,0.70,1082.0
ADV,0.22,0.07,0.11,174.0
AUX,0.00,0.00,0.00,51.0
CCONJ,0.62,0.08,0.14,193.0
DET,0.00,0.00,0.00,123.0
INTJ,0.00,0.00,0.00,0.0
NOUN,0.55,0.84,0.67,2463.0
NUM,0.00,0.00,0.00,155.0
PART,1.00,0.01,0.02,88.0


## Цель 3. Морфология: 10 признаков и многозадачная модель

**Выбор признаков.** В SynTagRus около 40 морфологических признаков. Берём 10 самых частых
(по числу токенов, у которых признак указан), чтобы не перегружать модель:

In [8]:
key_counts = collections.Counter(k for s in train_sents for _, _, f in s for k in f)
print('Признаки по частоте:', key_counts.most_common(15))

MORPH_KEYS = [k for k, _ in key_counts.most_common(N_MORPH)]
MORPH_VAL2ID = {}
for k in MORPH_KEYS:
    vals = sorted({f[k] for s in train_sents for _, _, f in s if k in f})
    MORPH_VAL2ID[k] = {v: i for i, v in enumerate(vals)}
print({k: len(v) for k, v in MORPH_VAL2ID.items()})

Признаки по частоте: [('Number', 21700), ('Case', 17878), ('Gender', 17648), ('Animacy', 12049), ('Degree', 5966), ('Aspect', 5685), ('VerbForm', 5685), ('Voice', 4801), ('Tense', 4741), ('Mood', 3909), ('Person', 2793), ('Variant', 379), ('Reflex', 80)]
{'Number': 2, 'Case': 7, 'Gender': 3, 'Animacy': 2, 'Degree': 3, 'Aspect': 2, 'VerbForm': 4, 'Voice': 2, 'Tense': 3, 'Mood': 2}


**Многозадачная модель.** Та же CNN, но поверх общего представления предложения стоит 11 выходов:
UPOS и по одному softmax на каждый из 10 признаков. Потери складываются с весами: UPOS — 1.0, каждый признак — 0.2.
Если признака у слова нет (в UD он `_`), метка `PAD_LABEL` не учитывается в потерях. Эта модель нужна для сравнения
с одной задачей: проверяем, помогает ли морфология части речи.

In [9]:
y_train_m = encode_targets(train_chunks, MORPH_KEYS)
y_val_m = encode_targets(val_chunks, MORPH_KEYS)
y_test_m = encode_targets(test_chunks, MORPH_KEYS)

cnn_multi, hist_multi = make_and_train('cnn_multi', 'cnn', morph=True)

scores['CNN многозадачная'] = evaluate_upos('CNN многозадачная', cnn_multi)
pd.DataFrame(scores).T.round(4)

Epoch 1/2


46/46 - 31s - 682ms/step - loss: 3.8651 - morph_Animacy_loss: 0.5627 - morph_Aspect_loss: 0.7335 - morph_Case_loss: 1.8011 - morph_Degree_loss: 0.3955 - morph_Gender_loss: 1.1031 - morph_Mood_loss: 0.2481 - morph_Number_loss: 0.6287 - morph_Tense_loss: 0.8848 - morph_VerbForm_loss: 1.0947 - morph_Voice_loss: 0.3810 - upos_loss: 2.2897 - upos_masked_accuracy: 0.2958 - val_loss: 3.0818 - val_morph_Animacy_loss: 0.5401 - val_morph_Aspect_loss: 0.6772 - val_morph_Case_loss: 1.6302 - val_morph_Degree_loss: 0.2048 - val_morph_Gender_loss: 1.0298 - val_morph_Mood_loss: 0.0572 - val_morph_Number_loss: 0.6111 - val_morph_Tense_loss: 0.7407 - val_morph_VerbForm_loss: 0.9050 - val_morph_Voice_loss: 0.2814 - val_upos_loss: 1.7461 - val_upos_masked_accuracy: 0.4445


Epoch 2/2


46/46 - 5s - 106ms/step - loss: 2.9208 - morph_Animacy_loss: 0.5452 - morph_Aspect_loss: 0.7278 - morph_Case_loss: 1.6740 - morph_Degree_loss: 0.1979 - morph_Gender_loss: 1.0905 - morph_Mood_loss: 0.0854 - morph_Number_loss: 0.5972 - morph_Tense_loss: 0.7241 - morph_VerbForm_loss: 0.8722 - morph_Voice_loss: 0.3144 - upos_loss: 1.5508 - upos_masked_accuracy: 0.4931 - val_loss: 2.4690 - val_morph_Animacy_loss: 0.5315 - val_morph_Aspect_loss: 0.6395 - val_morph_Case_loss: 1.5090 - val_morph_Degree_loss: 0.1895 - val_morph_Gender_loss: 1.0016 - val_morph_Mood_loss: 0.0453 - val_morph_Number_loss: 0.5772 - val_morph_Tense_loss: 0.5873 - val_morph_VerbForm_loss: 0.7015 - val_morph_Voice_loss: 0.2345 - val_upos_loss: 1.2559 - val_upos_masked_accuracy: 0.5693


Restoring model weights from the end of the best epoch: 2.


,точность,macro F1
CNN (одна задача),0.6159,0.2653
RNN (одна задача),0.5419,0.1856
CNN многозадачная,0.5761,0.2037


In [10]:
pred_m = predict_dict(cnn_multi, X_test)
morph_rows = []
for k in MORPH_KEYS:
    name = 'morph_' + k
    mask = y_test_m[name] != PAD_LABEL
    acc = (pred_m[name].argmax(-1)[mask] == y_test_m[name][mask]).mean()
    morph_rows.append({'признак': k, 'токенов с меткой': int(mask.sum()), 'точность': round(float(acc), 3)})
pd.DataFrame(morph_rows)

,признак,токенов с меткой,точность
0,Number,5256,0.763
1,Case,4496,0.369
2,Gender,4529,0.528
3,Animacy,3481,0.783
4,Degree,982,0.971
5,Aspect,1030,0.718
6,VerbForm,1030,0.737
7,Voice,926,0.874
8,Tense,871,0.726
9,Mood,742,0.995


In [11]:
with open(os.path.join(MODEL_DIR, 'vocab.json'), 'w', encoding='utf-8') as f:
    json.dump({'char2id': char2id, 'morph_keys': MORPH_KEYS, 'morph_values': MORPH_VAL2ID, 'upos': UPOS},
              f, ensure_ascii=False, indent=1)
print('Словарь сохранён:', os.path.join(MODEL_DIR, 'vocab.json'))

Словарь сохранён: C:\Users\reino\Desktop\Учеба\Homework\NLP\pos_tagging\models_smoke\vocab.json


### Функция разбора

Она нужна в целях 4 и 5. Для текста разбивает слова, прогоняет его через три модели и собирает таблицу.

In [12]:
id2val = {k: {i: v for v, i in MORPH_VAL2ID[k].items()} for k in MORPH_KEYS}


def analyze(text):
    tokens = TOKEN_RE.findall(text)
    rows = []
    for start in range(0, len(tokens), MAX_WORDS):
        chunk = tokens[start:start + MAX_WORDS]
        X = encode_forms([chunk])
        p_cnn = predict_dict(cnn_single, X)['upos'][0]
        p_rnn = predict_dict(rnn_single, X)['upos'][0]
        p_multi = predict_dict(cnn_multi, X)
        for j, word in enumerate(chunk):
            upos_m = UPOS[int(p_multi['upos'][0][j].argmax())]
            feats = []
            if upos_m not in ('PUNCT', 'SYM'):
                for k in MORPH_KEYS:
                    p = p_multi['morph_' + k][0][j]
                    if p.max() >= 0.5:
                        feats.append(f'{k}={id2val[k][int(p.argmax())]}')
            rows.append({'слово': word,
                         'CNN': UPOS[int(p_cnn[j].argmax())],
                         'CNN уверенность': round(float(p_cnn[j].max()), 2),
                         'RNN': UPOS[int(p_rnn[j].argmax())],
                         'CNN многозадачная': upos_m,
                         'признаки': '|'.join(feats)})
    return pd.DataFrame(rows)

## Цель 4. Проверка на омонимичных словах

**Что считаем.** Омонимичное слово — форма, которая в SynTagRus встречается с разными частями речи. Берём формы
с не менее чем 50 вхождениями, у которых минимальная доля второй части речи не меньше 15%
(например, «что» — PRON или SCONJ, «как» — ADV или SCONJ). Проверяем точность на этих словах в `nerus_lenta`.

Затем берём примеры из `ruscorpora_content.csv`, где омонимия снята вручную (колонка `Ambiguity`). Метки частей речи в этом файле
нет, поэтому проверка качественная: смотрим, какой тег модель даёт центральному слову.

In [13]:
upos_by_form = collections.defaultdict(collections.Counter)
for s in train_sents:
    for f, u, _ in s:
        upos_by_form[f.lower()][u] += 1

AMB = {f for f, c in upos_by_form.items()
       if sum(c.values()) >= 50 and len(c) >= 2 and min(c.values()) / sum(c.values()) >= 0.15}
top_amb = sorted(AMB, key=lambda f: -sum(upos_by_form[f].values()))[:10]
print('Омонимичных форм:', len(AMB))

flat = [(i, j, f.lower()) for i, ch in enumerate(test_chunks) for j, (f, _, _) in enumerate(ch)]
amb_pos = np.zeros(y_test['upos'].shape, dtype=bool)
for i, j, f in flat:
    if f in AMB:
        amb_pos[i, j] = True

rows = []
for form in top_amb:
    pos = [(i, j) for i, j, f in flat if f == form]
    if not pos:
        continue
    ii, jj = zip(*pos)
    row = {'форма': form,
           'разборы в train': ', '.join(f'{u}:{n}' for u, n in upos_by_form[form].most_common(3)),
           'токенов в nerus': len(pos)}
    for name, pred in results.items():
        row[name] = round(float((pred[ii, jj] == y_test['upos'][ii, jj]).mean()), 3)
    rows.append(row)
pd.DataFrame(rows)

Омонимичных форм: 9


,форма,разборы в train,токенов в nerus,CNN (одна задача),RNN (одна задача),CNN многозадачная
0,и,"CCONJ:919, PART:174",165,0.036,0.036,0.091
1,его,"DET:124, PRON:73",29,0.103,0.138,0.207
2,все,"DET:66, PRON:61, PART:27",9,0.000,0.000,0.000
3,было,"VERB:49, AUX:35",11,0.000,0.000,0.091
4,их,"DET:42, PRON:39",16,0.062,0.000,0.000
5,ее,"PRON:41, DET:33",10,0.000,0.100,0.000
6,был,"AUX:57, VERB:14",9,0.222,0.000,0.222
7,будет,"AUX:36, VERB:16",12,0.000,0.000,0.000
8,нет,"VERB:42, PART:8",2,0.000,0.000,0.000


In [14]:
summary = {}
for name, pred in results.items():
    summary[name] = {'все слова': round(float((pred[y_test['upos'] != PAD_LABEL] == y_test['upos'][y_test['upos'] != PAD_LABEL]).mean()), 3),
                     'омонимы': round(float((pred[amb_pos] == y_test['upos'][amb_pos]).mean()), 3)}
pd.DataFrame(summary).T

,все слова,омонимы
CNN (одна задача),0.616,0.046
RNN (одна задача),0.542,0.042
CNN многозадачная,0.576,0.091


In [15]:
with open(CSV_PATH, encoding='utf-8', newline='') as f:
    csv_rows = list(csv.DictReader(f, delimiter=';'))
manual = [r for r in csv_rows if r['Ambiguity'] == 'омонимия снята вручную' and r['Center'].lower() in AMB]
print('Примеров "омонимия снята вручную" с омонимичным словом:', len(manual))

Примеров "омонимия снята вручную" с омонимичным словом: 0


In [16]:
examples = []
for r in manual[:8]:
    tags = analyze(r['Full context'])
    hit = tags[tags['слово'].str.lower() == r['Center'].lower()]
    if len(hit):
        examples.append({'центральное слово': r['Center'],
                         'CNN': hit.iloc[0]['CNN'],
                         'CNN многозадачная': hit.iloc[0]['CNN многозадачная'],
                         'контекст': r['Full context'][:90]})
pd.DataFrame(examples)

""


## Цель 5. Разбор слова или предложения

Введите текст в переменную `text` и выполните ячейку. Можно передать одно слово или предложение.
Предложение длиннее 40 слов разбивается на части. Для каждого слова выводятся:

- часть речи по трём моделям (CNN, RNN, многозадачная CNN) и уверенность CNN;
- морфологические признаки от многозадачной модели (показываются признаки с вероятностью не ниже 0.5).

Модели берутся из памяти, если ноутбук уже выполнен в этой сессии, либо из кэша `models_syntagrus/`.
Функция `analyze` определена выше, в разделе «Функция разбора».

In [17]:
text = 'Вице-премьер по социальным вопросам вчера посетил больницу в Петербурге.'
analyze(text)

,слово,CNN,CNN уверенность,RNN,CNN многозадачная,признаки
0,Вице-премьер,NOUN,0.46,VERB,NOUN,Number=Sing|Animacy=Inan|Degree=Pos|Aspect=Imp...
1,по,ADP,0.58,ADP,ADP,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...
2,социальным,NOUN,0.33,VERB,VERB,Number=Sing|Gender=Masc|Animacy=Inan|Degree=Po...
3,вопросам,NOUN,0.60,NOUN,NOUN,Number=Sing|Gender=Masc|Animacy=Inan|Degree=Po...
4,вчера,NOUN,0.23,NOUN,NOUN,Number=Sing|Gender=Masc|Animacy=Inan|Degree=Po...
5,посетил,VERB,0.50,NOUN,VERB,Number=Sing|Gender=Masc|Animacy=Inan|Degree=Po...
6,больницу,NOUN,0.66,NOUN,NOUN,Number=Sing|Animacy=Inan|Degree=Pos|Aspect=Per...
7,в,ADP,0.67,ADP,ADP,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...
8,Петербурге,NOUN,0.64,NOUN,NOUN,Number=Sing|Animacy=Inan|Degree=Pos|Aspect=Imp...
9,.,PUNCT,1.00,PUNCT,PUNCT,


In [18]:
analyze('Мама')

,слово,CNN,CNN уверенность,RNN,CNN многозадачная,признаки
0,Мама,NOUN,0.36,NOUN,NOUN,Number=Sing|Animacy=Inan|Degree=Pos|Aspect=Imp...


In [19]:
text = 'Что делать, если пароль забыт?'
analyze(text)

,слово,CNN,CNN уверенность,RNN,CNN многозадачная,признаки
0,Что,PRON,0.26,PRON,PRON,Number=Sing|Case=Nom|Animacy=Inan|Degree=Pos|A...
1,делать,VERB,0.77,NOUN,NOUN,Number=Sing|Animacy=Inan|Degree=Pos|Aspect=Imp...
2,",",PUNCT,1.00,PUNCT,PUNCT,
3,если,ADV,0.19,ADV,VERB,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...
4,пароль,VERB,0.46,VERB,VERB,Number=Sing|Gender=Masc|Animacy=Inan|Degree=Po...
5,забыт,NOUN,0.51,NOUN,NOUN,Number=Sing|Animacy=Inan|Degree=Pos|Aspect=Imp...
6,?,PUNCT,1.00,PUNCT,PUNCT,
